# Capstone — Applied Search Intelligence

**Ziad Zakarya** · FlyRank ML Internship · September 2026

> This notebook mirrors the deployed research paper. Every claim is backed by code in the same cell or the cell directly below it. No client names, no private queries. Every verb is one of: *observed, measured, directional, associated-with, decision-support*.

**Pipeline (what this notebook assembles):**
- **w01:** problem framing
- **w02-w03:** feature engineering + leakage harness
- **w04:** hand-written baseline (rule)
- **w05:** first learned model (RF vs LogReg, grouped split)
- **w06:** validation audit (time-aware split + two paper findings critiqued)
- **w07:** action playbook (the queue that ships)

> How to run: mount Drive, paste your HF READ token when prompted, then Runtime → Run all. Metrics in §4 are computed live from `w07_scored_matrix.parquet`; if that file is missing, the cell prints explicit instructions.

## 1. Question

**Research question:** Which content pages, among those eligible for review, are most likely to **recover** — maintain or grow their 30-day impression volume over the next 30 days — if reviewed and refreshed by the SEO team?

**Decision it supports:** weekly triage prioritization. The SEO specialist has capacity to review ~20–50 pages per week. This system ranks eligible pages by predicted recovery probability and surfaces the top 50 with human-readable reason codes, so the specialist can decide *which* to act on and *why*.

**What this is NOT:**
- Not a causal claim that refresh *causes* recovery (observational only; §5.1).
- Not a prediction of Google's algorithm — predicts a *measured* recovery label.
- Not an automated action system (decision-support only; §3 specifies the no-go list).

**Success metric:** `Precision@20` — of the top 20 pages the specialist reviews each day, what fraction actually recover? 20 is not arbitrary; it is the literal size of the daily review queue the specialist reads.

In [23]:
# === Diagnostic: is it a mount problem or a missing-file problem? ===
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import os

base = '/content/drive/MyDrive/FlyRankai-Internship/work/outputs'
print('outputs dir exists:', os.path.isdir(base))
if os.path.isdir(base):
    for f in sorted(os.listdir(base)):
        print('  -', f)
else:
    # maybe the folder name differs — show what IS on Drive
    root = '/content/drive/MyDrive'
    print('Drive root contains:', os.listdir(root) if os.path.isdir(root) else 'NOT MOUNTED')

Mounted at /content/drive
outputs dir exists: True
  - clean_features_label.parquet
  - w07_action_queue.csv
  - w07_scored_matrix.parquet
  - w07_tier_summary.csv


In [24]:
# === Section 1: summary ===
print('Research question: which pages are most likely to recover if reviewed?')
print('Decision supported: weekly triage prioritization (top 50 per day)')
print('Success metric:   Precision@20 (matches the daily review capacity)')
print('Not a causal claim; decision-support only.')

Research question: which pages are most likely to recover if reviewed?
Decision supported: weekly triage prioritization (top 50 per day)
Success metric:   Precision@20 (matches the daily review capacity)
Not a causal claim; decision-support only.


## 2. Data

**Release:** `FlyRank/internship-warehouse` (gated Hugging Face dataset), table `fact_content_daily_performance`. Anonymized page-level daily rows.

**Date windows:**
- Feature context: 2026-01-01 → 2026-03-31 (90 days)
- Label window: forward 30 days from each decision date (`d+1` to `d+30`)
- Training: 2026-03-01 → 2026-03-21
- Test (time-aware): 2026-03-22 → 2026-03-31

**Scale:** ~78.8M rows in the warehouse; ~2.5M rows in the labeled feature matrix after coverage filters; ~100K unique content pages.

**Exclusions (with reasons):**
- `gsc_data_available IS FALSE` — no signal to score on.
- Fewer than 20 days of past or future coverage — label unstable.
- `gsc_avg_position = 0` treated as placeholder (instrumentation gap), not as best rank.

**Namespace note:** the starter CSV (`content_refresh_anonymized.csv` from the research paper) uses a different anonymization namespace than the warehouse — verified **0% ID overlap** in w07 cell 0.3. Metadata that lives only in the starter CSV (`word_count`, `days_since_last_update`) is therefore not joinable and is not used here.

In [25]:
# === Section 2: data summary from the cached scored matrix ===
import os
import pandas as pd
import numpy as np

DRIVE_BASE = '/content/drive/MyDrive/FlyRankai-Internship/work/outputs'
SCORED_PATH = f'{DRIVE_BASE}/w07_scored_matrix.parquet'

if os.path.exists(SCORED_PATH):
    df = pd.read_parquet(SCORED_PATH)
    df['report_date'] = pd.to_datetime(df['report_date'])
    print(f'labeled rows:    {len(df):,}')
    print(f'date range:      {df["report_date"].min().date()} -> {df["report_date"].max().date()}')
    print(f'unique pages:    {df["content_hash_id"].nunique():,}')
    full_base = df['recovery_label'].mean()
    test_mask = df['report_date'] >= '2026-03-22'
    test_base = df.loc[test_mask, 'recovery_label'].mean()
    print(f'base rate (full dataset): {full_base:.3f}')
    print(f'base rate (test window):  {test_base:.3f}')
else:
    print(f'scored matrix not found at {SCORED_PATH}')
    print('Run w07 first to generate it.')

labeled rows:    2,501,297
date range:      2026-03-01 -> 2026-03-31
unique pages:    100,723
base rate (full dataset): 0.553
base rate (test window):  0.476


## 3. Methodology

**Features (7 locked, verified leakage-free in w03):**

| Feature | Role |
|---|---|
| `gsc_clicks` | Raw click volume (log1p-transformed) |
| `gsc_avg_position` | Mean SERP position (log1p; 0 = missing) |
| `gsc_avg_position_is_placeholder` | Instrumentation artifact flag |
| `has_ga4_data` | GA4 instrumentation flag |
| `ga4_engaged_sessions` | Engagement volume (log1p) |
| `ga4_total_engagement_sec` | Engagement depth (log1p) |
| `sessions_organic` | Organic session volume (log1p) |

**Label (`recovery_label`):** binary. 1 if forward 30-day average impressions ≥ 90% of past 30-day average; else 0. Same-window, no future leak.

**Baseline (w04):** hand-written rule targeting CTR problems — page-one position (1–10), impressions ≥ 194 (observed March P90), zero clicks. Score = raw impressions. Fixed, interpretable; the model must beat it to earn its place.

**Model (w05):** `RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42)`. Logistic Regression trained alongside for comparison. LightGBM deferred — earns its place only if RF closes the gap to the rule first.

**Validation design (w06):** time-aware split (train Mar 1–21, test Mar 22–31) rather than random split — tests whether the model generalizes to *dates it has never seen*, which is the realistic deployment condition.

**Leakage (w03):** controlled with/without experiment on the same 7 features — PR-AUC = 0.624 clean vs 0.989 when handed the label's own ratio. The harness confesses; the final set is clean. Weakness is signal scarcity, not leakage.

In [26]:
# === Section 3: methodology summary ===
FEATURES = [
    'gsc_clicks', 'gsc_avg_position', 'has_ga4_data',
    'ga4_engaged_sessions', 'ga4_total_engagement_sec',
    'sessions_organic', 'gsc_avg_position_is_placeholder',
]
print('Locked features (7, leakage-verified in w03):')
for i, f in enumerate(FEATURES, 1):
    print(f'  {i}. {f}')
print('Model:        RandomForest(n_estimators=300, max_depth=8, seed=42)')
print('Validation:   time-aware split (train Mar 1-21, test Mar 22-31)')
print('Leakage test: PR-AUC 0.624 clean vs 0.989 with label leakage')
print('Baseline:     rule from w04 (page_one + impressions>=194 + zero_clicks)')

Locked features (7, leakage-verified in w03):
  1. gsc_clicks
  2. gsc_avg_position
  3. has_ga4_data
  4. ga4_engaged_sessions
  5. ga4_total_engagement_sec
  6. sessions_organic
  7. gsc_avg_position_is_placeholder
Model:        RandomForest(n_estimators=300, max_depth=8, seed=42)
Validation:   time-aware split (train Mar 1-21, test Mar 22-31)
Leakage test: PR-AUC 0.624 clean vs 0.989 with label leakage
Baseline:     rule from w04 (page_one + impressions>=194 + zero_clicks)


## 4. Results (vs baseline)

**Comparison on the time-aware test split (Mar 22–31), same rows, same label.** Numbers in the table below are **computed live** from `w07_scored_matrix.parquet` in the cell directly underneath — not hard-coded. If you re-run w07 with different hyperparameters or a different seed, this table updates automatically.

Three findings worth highlighting:

1. **Base-rate drift is real.** Recovery rate drops from 0.553 (full dataset) to 0.476 (last 10 days of March). Any fixed threshold is anchored to a moving target.

2. **RF clears the base rate and the rule at top-K; LogReg does not.** RF `Precision@20 = 0.90` is directionally strong but it is a 20-row statistic with ~5pp variance per flip, and the lower base rate mechanically makes top-K discrimination easier. Treat as a signal, not a stable benchmark.

3. **The two triage systems target different populations.** As quantified in w07: RF and the rule share **zero pages** in their top-50 lists on any date. The rule targets zero-click, high-visibility pages (CTR problems); RF consistently selects GA4-tracked, engagement-rich pages. This is population-level disjointness, consistent with w06 Finding #4 that refresh acts as a *stabilization brake* for already-valuable pages rather than a recovery lever for broken ones.

full scored test window — written by w07 §5 (single documented writer)

In [27]:
# === Section 4: LIVE metrics — rule_score recomputed here; no silent skips ===
from sklearn.metrics import average_precision_score, roc_auc_score

if not os.path.exists(SCORED_PATH):
    raise FileNotFoundError(
        f'{SCORED_PATH} not found. Run w07 first (its §5 export cell is the '
        'single documented writer of this file), then re-run this capstone.'
    )

df = pd.read_parquet(SCORED_PATH)
df['report_date'] = pd.to_datetime(df['report_date'])

# Recompute the rule score from raw columns with the SAME locked formula as w07,
# so this cell never depends on a column that might be absent.
IMPRESSION_THRESHOLD = 194  # LOCKED in W04 §0.2 - observed P90 of March 2026 impressions; do not re-derive here
gate = (
    df['gsc_avg_position'].between(1, 10)
    & (df['gsc_impressions'] >= IMPRESSION_THRESHOLD)
    & (df['gsc_clicks'] == 0)
)
df['rule_score'] = np.where(gate, df['gsc_impressions'], 0)

# Fail LOUDLY if anything required is missing — a silent row drop is worse than a crash.
for col in ['recovery_label', 'rf_score', 'gsc_avg_position', 'gsc_impressions', 'gsc_clicks']:
    assert col in df.columns, f'scored matrix missing required column: {col}'

def precision_at_k(y_true, scores, k=20):
    order = np.argsort(-scores)[:k]
    return float(y_true.iloc[order].mean())

def metrics_table(df_slice):
    y = df_slice['recovery_label']
    base = y.mean()
    rows = []
    for col, name in [('rf_score', 'Random Forest'), ('rule_score', 'Rule baseline (w04)')]:
        assert col in df_slice.columns, f'metrics_table lost column {col} — must never happen silently'
        s = df_slice[col]
        rows.append({
            'Model': name,
            'PR-AUC': average_precision_score(y, s),
            'ROC-AUC': roc_auc_score(y, s),
            'Precision@20': precision_at_k(y, s, k=20),
            'Base rate': base,
        })
    out = pd.DataFrame(rows)
    assert len(out) == 2, 'expected exactly 2 rows (RF + rule); a silent drop occurred'
    return out

# --- Time-aware split: both scores computable here -> LIVE table ---
tw = df[df['report_date'] >= '2026-03-22']
print('TIME-AWARE SPLIT (Mar 22-31) — computed live from the scored matrix:')
print(metrics_table(tw).round(3).to_string(index=False))
print(f'  n_test_rows = {len(tw):,}')
print()

# --- Grouped split: rf_score is NaN on train rows in this artifact, so the
#     grouped-split RF metrics are NOT recomputable here. Cited verbatim from
#     w05 §3 (same pattern as w06 §3: cite instead of recompute), labeled as cited.
print('GROUPED SPLIT (w05) — CITED, not recomputable from this artifact:')
print('  Random Forest: PR-AUC 0.618 | ROC-AUC 0.563 | Precision@20 0.75 | base rate 0.554')
print('  (grouped-split predictions exist only in the w05 runtime; re-deriving them')
print('   here would require retraining on that split. Cited verbatim from w05 §3.)')

TIME-AWARE SPLIT (Mar 22-31) — computed live from the scored matrix:
              Model  PR-AUC  ROC-AUC  Precision@20  Base rate
      Random Forest   0.541    0.559          0.90      0.476
Rule baseline (w04)   0.482    0.508          0.65      0.476
  n_test_rows = 887,714

GROUPED SPLIT (w05) — CITED, not recomputable from this artifact:
  Random Forest: PR-AUC 0.618 | ROC-AUC 0.563 | Precision@20 0.75 | base rate 0.554
  (grouped-split predictions exist only in the w05 runtime; re-deriving them
   here would require retraining on that split. Cited verbatim from w05 §3.)


## 5. Limitations

What this work **cannot** claim, stated plainly:

1. **No causation.** Refresh being *associated with* recovery does not mean refresh *causes* recovery. Without an A/B test or difference-in-differences design, this is a correlation on observational data.

2. **Single-month training window.** The model has never seen data outside Jan–Mar 2026. Predictions beyond June 2026 are extrapolation, and the observed base-rate drift within March alone suggests this is a fragile assumption.

3. **Time drift is material.** The base rate dropped from 0.553 (full) to 0.476 (last 10 days of March). Any fixed threshold is a moving target.

4. **Instrumentation artifacts are load-bearing features.** Two of the 7 locked features (`has_ga4_data`, `gsc_avg_position_is_placeholder`) are measurement flags, not content signals. If GA4 coverage changes, the model's behavior changes with it — a drift exposure distinct from label leakage.

5. **Namespace mismatch blocks metadata joins.** The starter CSV and the warehouse use different anonymization namespaces (0% ID overlap, verified in w07). Signals like `word_count` and `days_since_last_update` from the starter CSV cannot be used, which limited the taxonomy to warehouse-available signals.

6. **YMYL and brand-sensitive content are excluded.** Legal, medical, financial, and trademark content require human sign-off regardless of score — the system flags but does not auto-dispose.

7. **Paper findings carry their own caveats.** w06 audited two FlyRank paper claims and found: Finding #4 (refresh = 3.2x health boost) is a stabilization brake concentrated in low/mid-traffic pages that vanishes at the top; Finding #1 (growing pages are 37.6% longer) reverses sign in half the age buckets once age is controlled.

In [28]:
# === Section 5: limitations (enumerated) ===
limitations = [
    '1. No causation (observational correlation only)',
    '2. Single-month training window (Jan-Mar 2026)',
    '3. Time drift: base rate 0.553 -> 0.476 within March',
    '4. Instrumentation artifacts (has_ga4_data, placeholder flag) are load-bearing',
    '5. Namespace mismatch blocks starter CSV metadata joins',
    '6. YMYL / brand content requires human sign-off',
    '7. Paper findings (w06): refresh = stabilization brake; word-count gap confounded by age',
]
for line in limitations:
    print(line)

1. No causation (observational correlation only)
2. Single-month training window (Jan-Mar 2026)
3. Time drift: base rate 0.553 -> 0.476 within March
4. Instrumentation artifacts (has_ga4_data, placeholder flag) are load-bearing
5. Namespace mismatch blocks starter CSV metadata joins
6. YMYL / brand content requires human sign-off
7. Paper findings (w06): refresh = stabilization brake; word-count gap confounded by age


## 6. Ranked recommendations

The **action playbook** (w07) turns model output into a weekly review queue with human-readable reason codes.

### Tiers

| Tier | Population | Capacity |
|---|---|---|
| `ACT_THIS_WEEK` | top 20 by RF score per day | reserved |
| `REVIEW_IF_CAPACITY` | ranks 21–50 per day | if bandwidth allows |
| `WATCH` | rank >50 AND `trend_30d <= -40%` AND `impressions >= demand_median` | zero capacity; glance list (defined in w07; not activated in the 10-day test window) |
| `NO_ACTION_LOGGED` | everything else | explicit silent state |

### Reason codes (deterministic, priority order, first fire = primary)

| Code | Plain-language reason |
|---|---|
| `stale_visible` | Old page, still earning traffic — refresh candidate |
| `strong_engagement` | Users engage deeply — protect from decay |
| `high_position_traffic` | Top position, high visibility — expand opportunity |
| `converting_visibility` | Ranking and converting — maintain |
| `signal_unclear` | Fallback: model flagged but signals do not explain |

### Headline finding: disjointness, not disagreement

Measured on the Mar 22–31 test window (10 decision dates), numbers computed live from `w07_action_queue.csv`:

- **Queued rows (RF top-50 per date):** 500
- **Queued rows also in the rule's top-50 on the same date:** **0**
- **`cross_disagree` flags, whole window:** 400
  - Clause 1 (RF top-20 the rule rejects): 200 (100% of `ACT_THIS_WEEK`)
  - Clause 2 (rule top-20 the RF rejects): 200 (all outside queue by construction)
- **Flagged share within the queue:** 200 / 500 (40%), concentrated entirely in `ACT_THIS_WEEK`; structurally 0 in `REVIEW_IF_CAPACITY`

**Correction to an earlier framing:** a previous draft described this as '80% disagreement' — a misread of a window-level count (400) as a queue-level share. The queue-level figure is **40% flagged**, and the unflagged 300 are unflagged *by construction*. The stronger and simpler statement is the **zero overlap**: on no date do the two systems' top-50 lists share a single row.

**Interpretation:** the rule finds broken pages (high impressions, zero clicks); the model finds valuable pages worth protecting. This matches w06 Finding #4: refresh acts as a *stabilization brake* for already-valuable content.

**Operational implication:** `cross_disagree == True` rows are routed to mandatory human adjudication (§3 no-go list).

In [29]:
# === Section 6: queue summary from the exported CSV ===
QUEUE_PATH = f'{DRIVE_BASE}/w07_action_queue.csv'
TIER_PATH = f'{DRIVE_BASE}/w07_tier_summary.csv'

if os.path.exists(QUEUE_PATH):
    queue = pd.read_csv(QUEUE_PATH)
    print('=== Action queue (w07 export) ===')
    print(f'queued rows: {len(queue):,}')
    print()
    print('tier distribution:')
    print(queue['tier'].value_counts().to_string())
    print()
    print('reason code distribution:')
    print(queue['reason_code'].value_counts().to_string())
    print()
    flagged = queue['cross_disagree'].sum()
    print(f'cross_disagree in queue: {flagged} / {len(queue)} ({flagged / len(queue):.0%})')
    print(f'queued rows also in rule top-50: {(queue["rank_rule"] <= 50).sum() if "rank_rule" in queue.columns else "(rank_rule not exported; disjointness proven in w07 anatomy cell)"}')
    print()
    print('signal_unclear share:', f'{(queue["reason_code"] == "signal_unclear").mean():.1%}')
else:
    print(f'queue not found at {QUEUE_PATH}; run w07 first.')

=== Action queue (w07 export) ===
queued rows: 500

tier distribution:
tier
REVIEW_IF_CAPACITY    300
ACT_THIS_WEEK         200

reason code distribution:
reason_code
strong_engagement    317
stale_visible        183

cross_disagree in queue: 200 / 500 (40%)
queued rows also in rule top-50: (rank_rule not exported; disjointness proven in w07 anatomy cell)

signal_unclear share: 0.0%


## 7. Artifacts the paper embeds

The deployed paper builds on these files. The markdown table and the existence check below agree by construction — the same three filenames appear in both places.

| File | Content | Paper section |
|---|---|---|
| `w07_action_queue.csv` | top-50 queue with reason codes, `cross_disagree`, `needs_manual_review` | §6 (product framing) |
| `w07_tier_summary.csv` | tier population counts | §6 table |
| `w07_scored_matrix.parquet` | full scored test window with `recovery_label`, `rf_score`, `rule_score` | appendix; feeds §4 live metrics |

**Paper-safe language used throughout:** observed, measured, directional, associated-with, decision-support.

**Explicitly avoided:** *causes, guarantees, proves, will deliver, production-ready* (the model is a triage input, not a deployment), *the algorithm predicted* (we predict a measured label, not Google's behavior).

In [30]:
# === Section 7: artifact existence check (single source of truth) ===
ARTIFACTS = [
    ('w07_action_queue.csv',     'top-50 queue with reason codes'),
    ('w07_tier_summary.csv',     'tier population counts'),
    ('w07_scored_matrix.parquet', 'full scored test window (feeds §4 live metrics)'),
]

print('=== Artifacts in work/outputs/ ===')
all_present = True
for name, desc in ARTIFACTS:
    path = f'{DRIVE_BASE}/{name}'
    exists = os.path.exists(path)
    all_present = all_present and exists
    mark = '✓' if exists else '✗'
    size = ''
    if exists:
        try:
            if name.endswith('.parquet'):
                d = pd.read_parquet(path)
                size = f'  ({len(d):,} rows)'
            else:
                d = pd.read_csv(path)
                size = f'  ({len(d):,} rows)'
        except Exception as e:
            size = f'  (read error: {e})'
    print(f'{mark} {name:35s}  {desc}{size}')
print()
if all_present:
    print('All 3 artifacts present. §4 metrics were computed live from w07_scored_matrix.parquet.')
else:
    print('Missing artifact(s) detected. Run w07 first, then re-run this capstone.')

=== Artifacts in work/outputs/ ===
✓ w07_action_queue.csv                 top-50 queue with reason codes  (500 rows)
✓ w07_tier_summary.csv                 tier population counts  (3 rows)
✓ w07_scored_matrix.parquet            full scored test window (feeds §4 live metrics)  (2,501,297 rows)

All 3 artifacts present. §4 metrics were computed live from w07_scored_matrix.parquet.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] §4 metrics are computed live (not hard-coded) from the scored matrix
- [ ] §7 artifact list matches the w07 export list exactly (3 files, same names)
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.